# Uncertainty-Aware Segmentation in Musculoskeletal Ultrasound
## A pipeline study in preparation for synovitis segmentation in arthritis ultrasound

**Author:** Bhagyaprasad Vastrad
**Date:** October 2026
**Notebook 1 of 2:** data preparation and subject-level splits

---

## About this project

I want to find out whether uncertainty estimation can tell me when a segmentation model's output on ultrasound images should not be trusted. My long-term target is synovitis segmentation in arthritis ultrasound. I do not have a public synovitis dataset, so I am building and testing my full pipeline on a public musculoskeletal ultrasound dataset first. I plan to apply the same pipeline to synovitis data under the data governance of a research group that holds such data.

**My research questions**
1. How accurately can a U-Net segment expert-annotated structures in ultrasound images?
2. Can uncertainty estimates identify the images where the segmentation is more likely to be wrong?
3. How do accuracy and uncertainty change when the data distribution shifts?

**Dataset.** Marzola et al. (2021): transverse ultrasound images of three muscles (biceps brachii BB, gastrocnemius medialis GM, tibialis anterior TA), healthy and pathological, with expert manual segmentation masks. It holds 8,169 images. All images were acquired at one centre (Radboud University Medical Center) on one scanner (Esaote MyLab Twice).

Citation: Marzola F, van Alfen N, Doorduin J, Meiburger KM. Deep learning segmentation of transverse musculoskeletal ultrasound images for neuromuscular disease assessment. Computers in Biology and Medicine, 2021, 104623.

## What I can and cannot claim from this dataset

- I can study segmentation accuracy and how well uncertainty tracks error.
- I can use two shifts as proxies for distribution shift: muscle type (for example train on BB and GM, test on TA) and disease status (train on healthy, test on pathological).
- I cannot test scanner or centre shift, because there is only one scanner and one centre.
- The targets are large (about a fifth of each image), unlike small synovitis regions, so empty-mask and boundary-metric behaviour will not be fully exercised.
- This is muscle, not synovitis, so I will not present these results as synovitis results.

## What this notebook does

1. Lists every image and checks that each has a mask.
2. Splits the data by subject, so images of one person never appear in both training and testing.
3. Checks the masks and the images for problems.
4. Saves `index.csv` and a compact `data_256.npz` that I upload to Colab for training.

## 1. Setup

In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image

# I point ROOT at the folder where I extracted the dataset.
# This is the only line I should need to edit.
ROOT = r"D:\0001_Kings_AI_Project\medical_imaging_ai\data\Polito-Radboud-DeepLearningUS"

OUT = "index.csv"      # the master table this notebook creates
SEED = 42              # I fix the seed so my split can be reproduced exactly
VAL_FRAC = 0.10        # share of subjects held out for validation
TEST_FRAC = 0.20       # share of subjects held out for testing

MUSCLES = ["BB", "GM", "TA"]
CONDITIONS = ["Healthy", "Pathological"]

## 2. Listing every image and checking that each has a mask

File names look like `anon_<subject>_<image number>.png`, so one subject contributes several images. I build one table with a row per image and stop with an error if any image has no mask.

In [3]:
rows = []
for m in MUSCLES:
    for c in CONDITIONS:
        img_dir = os.path.join(ROOT, m, c, "Images")
        msk_dir = os.path.join(ROOT, m, c, "Masks")
        img_files = sorted(os.listdir(img_dir))
        msk_files = set(os.listdir(msk_dir))

        # I want every image to have a mask with the identical file name
        missing = [f for f in img_files if f not in msk_files]
        if missing:
            raise RuntimeError(f"{m}/{c}: {len(missing)} images have no mask, e.g. {missing[:3]}")

        for f in img_files:
            rows.append(dict(
                muscle=m, condition=c,
                subject=f.split("_")[1],      # the number between the underscores
                fname=f,
                img_path=os.path.join(img_dir, f),
                mask_path=os.path.join(msk_dir, f),
            ))

df = pd.DataFrame(rows)
print(f"Images: {len(df)}   Unique subject IDs: {df.subject.nunique()}")

Images: 8169   Unique subject IDs: 1363


## 3. Subject-level split

I split by subject, not by image, because images from the same person look alike and a random image-level split would leak information into the test set.

The same ID appears in more than one muscle folder. I cannot confirm that an ID means the same person across muscles (the overlap between folders is close to what chance would give), so I treat each ID as one person everywhere. This can only separate subjects more than necessary, never less, so it cannot cause leakage.

In [4]:
subjects = np.array(sorted(df.subject.unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(subjects)

n = len(subjects)
n_test = int(round(TEST_FRAC * n))
n_val = int(round(VAL_FRAC * n))

# I give each subject exactly one split, shared across all muscles
split_of = {}
for s in subjects[:n_test]:
    split_of[s] = "test"
for s in subjects[n_test:n_test + n_val]:
    split_of[s] = "val"
for s in subjects[n_test + n_val:]:
    split_of[s] = "train"
df["split"] = df.subject.map(split_of)

# Leakage check: no subject may appear in more than one split
assert (df.groupby("subject").split.nunique() == 1).all(), "a subject appears in more than one split"
print("Leakage check passed. Subjects per split:")
print(df.groupby("split").subject.nunique())

Leakage check passed. Subjects per split:
split
test     273
train    954
val      136
Name: subject, dtype: int64


## 4. Checking the masks

I measure how much of each image the mask covers. This tells me whether any masks are empty and how small the smallest targets are. It matters later, because empty masks need special handling in the Dice score and Hausdorff distance.

In [5]:
fg = []
for i, p in enumerate(df.mask_path):
    mk = np.asarray(Image.open(p).convert("L"))
    fg.append(float((mk > 127).mean()))     # fraction of pixels inside the mask
    if (i + 1) % 2000 == 0:
        print(f"  checked {i + 1}/{len(df)} masks")
df["fg_fraction"] = fg
print("Done.")

  checked 2000/8169 masks
  checked 4000/8169 masks
  checked 6000/8169 masks
  checked 8000/8169 masks
Done.


## 5. Other data checks

Two things I want to know before training:
- Whether any subject ID is labelled Healthy in one muscle and Pathological in another.
- Whether the BB images, which load as RGB, contain real colour information or are just grayscale stored in three channels.

In [6]:
cond_per_subject = df.groupby("subject").condition.nunique()
mixed = int((cond_per_subject > 1).sum())

# I check a random sample of 100 BB images for colour information
bb_all = df[df.muscle == "BB"]
bb = bb_all.sample(n=min(100, len(bb_all)), random_state=0)
non_gray = 0
for p in bb.img_path:
    a = np.asarray(Image.open(p).convert("RGB")).astype(int)
    if np.abs(a[..., 0] - a[..., 1]).max() > 0 or np.abs(a[..., 1] - a[..., 2]).max() > 0:
        non_gray += 1

print(f"Empty masks (of {len(df)}): {int((df.fg_fraction == 0).sum())}")
print(f"Masks covering under 2% of the image: {int((df.fg_fraction < 0.02).sum())}")
print(f"Subject IDs labelled both Healthy and Pathological across muscles: {mixed}")
print(f"BB images with colour information in the sampled 100: {non_gray}")

Empty masks (of 8169): 0
Masks covering under 2% of the image: 50
Subject IDs labelled both Healthy and Pathological across muscles: 325
BB images with colour information in the sampled 100: 0


## 6. Saving the index and printing the report

In [8]:
df.to_csv(OUT, index=False)
print(f"Saved {OUT}\n")

print("=== Images per split / muscle / condition ===")
print(df.pivot_table(index=["muscle", "condition"], columns="split",
                     values="fname", aggfunc="count").fillna(0).astype(int))

print("\n=== Mask coverage by muscle / condition (fraction of image) ===")
print(df.groupby(["muscle", "condition"]).fg_fraction
        .describe()[["count", "min", "mean", "max"]].round(3))

Saved index.csv

=== Images per split / muscle / condition ===
split                test  train  val
muscle condition                     
BB     Healthy        473   1499  235
       Pathological    84    400   43
GM     Healthy        315   1010  155
       Pathological    55    252   43
TA     Healthy        618   2167  307
       Pathological    98    360   55

=== Mask coverage by muscle / condition (fraction of image) ===
                      count    min   mean    max
muscle condition                                
BB     Healthy       2207.0  0.000  0.195  0.455
       Pathological   527.0  0.017  0.157  0.348
GM     Healthy       1480.0  0.065  0.204  0.412
       Pathological   350.0  0.017  0.156  0.364
TA     Healthy       3092.0  0.000  0.216  0.481
       Pathological   513.0  0.013  0.169  0.346


## 7. Exporting compact 256 x 256 arrays for Colab

Training on a CPU is far too slow, so I will train on a GPU in Google Colab. I do not want to upload 3 GB of PNG files, so I save every image and mask resized to 256 x 256 into one file, `data_256.npz`, in the same row order as `index.csv`. I upload only these two files to Colab.

In [9]:
SIZE = 256
imgs = np.zeros((len(df), SIZE, SIZE), dtype=np.uint8)
msks = np.zeros((len(df), SIZE, SIZE), dtype=np.uint8)

for i, r in enumerate(df.itertuples()):
    # bilinear resizing for the image, nearest-neighbour for the mask so it stays binary
    imgs[i] = np.asarray(Image.open(r.img_path).convert("L").resize((SIZE, SIZE), Image.BILINEAR))
    msks[i] = np.asarray(Image.open(r.mask_path).convert("L").resize((SIZE, SIZE), Image.NEAREST)) > 127
    if (i + 1) % 2000 == 0:
        print(f"  exported {i + 1}/{len(df)}")

np.savez_compressed("data_256.npz", imgs=imgs, msks=msks)
print("Saved data_256.npz,", round(os.path.getsize("data_256.npz") / 1e6), "MB")

  exported 2000/8169
  exported 4000/8169
  exported 6000/8169
  exported 8000/8169
Saved data_256.npz, 425 MB
